In [0]:
# Install the vector search SDK
%pip install databricks-vectorsearch

# Restart Python to ensure the package is available
dbutils.library.restartPython()

  Obtaining dependency information for databricks-vectorsearch from https://files.pythonhosted.org/packages/43/ab/4d6ba34ff89e326cd867e3d1ccf503a2e826558942a36ab3da34e1970ae7/databricks_vectorsearch-0.63-py3-none-any.whl.metadata
  Obtaining dependency information for deprecation>=2 from https://files.pythonhosted.org/packages/02/c3/253a89ee03fc9b9682f1541728eb66db7db22148cd94f89ab22528cd1e1b/deprecation-2.1.0-py2.py3-none-any.whl.metadata
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


In [0]:
#configuration

VS_ENDPOINT_NAME = "brd-agent-vs-endpoint"

EMBEDDING_MODEL = "databricks-bge-large-en"

NOTEBOOK_SOURCE_TABLE = "main.doc_test.notebook_chunks"
BRD_SOURCE_TABLE      = "main.doc_test.brd_chunks"

NOTEBOOK_INDEX_NAME = "main.doc_test.brd_notebook_index"
BRD_INDEX_NAME      = "main.doc_test.brd_gold_brd_index"

print("Config loaded")


Config loaded


In [0]:
# create or get vector search endpoint

try:
    existing_eps = [ep for ep in vsc.list_endpoints()]

    if VS_ENDPOINT_NAME not in existing_eps:
        vsc.create_endpoint(
            name=VS_ENDPOINT_NAME,
            endpoint_type="STANDARD"
        )
        print(f"Created Vector Search endpoint: {VS_ENDPOINT_NAME}")
    else:
        print(f"Vector Search endpoint already exists: {VS_ENDPOINT_NAME}")

except Exception as e:
    # If the error is that the endpoint already exists, handle it gracefully
    if "ALREADY_EXISTS" in str(e):
        print(f"Vector Search endpoint '{VS_ENDPOINT_NAME}' already exists (caught exception).")
    else:
        # Re-raise unexpected exceptions
        raise

Vector Search endpoint 'brd-agent-vs-endpoint' already exists (caught exception).


In [0]:
%sql  -- need this for table uc registration, uc metadata registration
CREATE SCHEMA IF NOT EXISTS main.doc_test;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS main.doc_test.notebook_chunks (
  chunk_id STRING,
  notebook_name STRING,
  schema STRING,
  domain STRING,
  chunk_type STRING,
  cell_index INT,
  content STRING
)
USING DELTA;


In [0]:
%sql
-- brd chunks table creation

CREATE TABLE IF NOT EXISTS main.doc_test.brd_chunks (
  chunk_id STRING,
  brd_name STRING,
  section STRING,
  domain STRING,
  content STRING
)
USING DELTA;


In [0]:
(
    df_nb
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.doc_test.brd_chunks")
)


In [0]:
#load data into table from volume (1 time)

df_nb = spark.read.format("delta").load(
    "/Volumes/tmp/tmp/brd_agent/delta/notebook_chunks"
)

df_nb.write.mode("append").saveAsTable(
    "main.doc_test.notebook_chunks"
)


In [0]:
# for brd chunks - loading into uc table

df_brd = spark.read.format("delta").load(
    "/Volumes/tmp/tmp/brd_agent/delta/brd_chunks"
)

(
    df_brd
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.doc_test.brd_chunks")
)


In [0]:
%sql
SELECT COUNT(*) FROM main.doc_test.notebook_chunks union all
SELECT COUNT(*) FROM main.doc_test.brd_chunks;


count(1)
668
165


In [0]:
%sql
-- got duplicate rowsw due to append - test and train

SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT chunk_id) AS distinct_chunks
FROM main.doc_test.notebook_chunks;


total_rows,distinct_chunks
668,334


In [0]:
# deduplication

from pyspark.sql import functions as F

df_nb = spark.table("main.doc_test.notebook_chunks")

df_nb_dedup = (
    df_nb
    .dropDuplicates(["chunk_id"])
)

df_nb_dedup.count()


334

In [0]:
(
    df_nb_dedup
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.doc_test.notebook_chunks")
)


In [0]:
%sql  -- have to enable change data feed
ALTER TABLE main.doc_test.notebook_chunks
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);


In [0]:
%sql  -- have to enable change data feed
ALTER TABLE main.doc_test.brd_chunks
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);


In [0]:
#creating notebook vector index

vsc.create_delta_sync_index(
    endpoint_name=VS_ENDPOINT_NAME,
    source_table_name=NOTEBOOK_SOURCE_TABLE,
    index_name=NOTEBOOK_INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="content",
    embedding_model_endpoint_name=EMBEDDING_MODEL,
    columns_to_sync=[
        "notebook_name",
        "schema",
        "domain",
        "chunk_type",
        "cell_index"
    ]
)

print("Notebook vector index created")


Notebook vector index created


In [0]:
# creating brd gold vector index

vsc.create_delta_sync_index(
    endpoint_name=VS_ENDPOINT_NAME,
    source_table_name=BRD_SOURCE_TABLE,
    index_name=BRD_INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="content",
    embedding_model_endpoint_name=EMBEDDING_MODEL,
    columns_to_sync=[
        "brd_name",
        "section",
        "domain"
    ]
)

print("BRD vector index created")


BRD vector index created


In [0]:
# Get the index object and trigger sync for each index
notebook_index = vsc.get_index(index_name=NOTEBOOK_INDEX_NAME)
notebook_index.sync()

brd_index = vsc.get_index(index_name=BRD_INDEX_NAME)
brd_index.sync()

print("Vector index sync triggered")


Vector index sync triggered


In [0]:
# validate 
display(vsc.list_indexes(VS_ENDPOINT_NAME))


{'vector_indexes': [{'name': 'main.doc_test.brd_gold_brd_index',
   'endpoint_name': 'brd-agent-vs-endpoint',
   'primary_key': 'chunk_id',
   'index_type': 'DELTA_SYNC',
   'creator': 'shivam.kumar@dvn.com',
   'id': '1b44258c-2905-47f1-9cdb-78fad7457113'},
  {'name': 'main.doc_test.brd_notebook_index',
   'endpoint_name': 'brd-agent-vs-endpoint',
   'primary_key': 'chunk_id',
   'index_type': 'DELTA_SYNC',
   'creator': 'shivam.kumar@dvn.com',
   'id': '385b1b5c-fcfd-4b61-b199-50ab23d46040'}]}